In [ ]:
import numpy as np
from scipy.stats import norm
import time
import pandas as pd
from joblib import Parallel, delayed

# 1. Absolute Differences (Monte Carlo Approximation)
def crps_absolute_differences_parallel(y, sample):
    M = len(sample)
    term1 = np.mean(np.abs(sample - y))
    
    # Parallelized computation of the pairwise absolute differences
    def pairwise_abs_diff(i, sample):
        return np.sum(np.abs(sample[i] - sample))
    
    term2 = 0.5 * np.sum(Parallel(n_jobs=-1)(delayed(pairwise_abs_diff)(i, sample) for i in range(M))) / M**2
    return term1 - term2

# 2. Quantile Representation (Monte Carlo Approximation)
def crps_quantile_representation(y, sample):
    M = len(sample)
    sorted_sample = np.sort(sample)
    tau_values = np.linspace(1 / M, 1, M)
    crps_value = 0
    for k in range(M):
        x_k = sorted_sample[k]
        tau_k = tau_values[k]
        if y >= x_k:
            crps_value += tau_k * (y - x_k)
        else:
            crps_value += (1 - tau_k) * (x_k - y)
    return 2 * crps_value / M

# 3. Corrected Theoretical Closed-Form CRPS for Normal Distribution
def crps_closed_form_normal(y):
    phi_y = norm.pdf(y)
    Phi_y = norm.cdf(y)
    term1 = 2 * phi_y + y * (2 * Phi_y - 1)
    term2 = np.sqrt(1 / np.pi)
    return term1 - term2

# Generate an observation from a standard normal distribution
y_observation = 0

# Generate a sample from a standard normal distribution for approximation purposes
sample_size = 100000
sample = np.random.normal(size=sample_size)

# Measure time and compute CRPS for each method
# Absolute Differences Approximation (Parallelized)
start_time = time.time()
crps_abs_diff = crps_absolute_differences_parallel(y_observation, sample)
abs_diff_time = time.time() - start_time

# Quantile Representation Approximation
start_time = time.time()
crps_quantile = crps_quantile_representation(y_observation, sample)
quantile_time = time.time() - start_time

# Closed-Form CRPS for Standard Normal
start_time = time.time()
crps_closed_form = crps_closed_form_normal(y_observation)
closed_form_time = time.time() - start_time

# Create a DataFrame to display results as a table
results = pd.DataFrame({
    'Method': ['Absolute Differences (Parallel)', 'Quantile Representation', 'Closed Form'],
    'CRPS Value': [crps_abs_diff, crps_quantile, crps_closed_form],
    'Time (seconds)': [abs_diff_time, quantile_time, closed_form_time]
})

# Print the table
print(results)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm
from joblib import Parallel, delayed

# Define functions for CRPS calculations
def crps_absolute_differences_parallel(y, sample):
    M = len(sample)
    term1 = np.mean(np.abs(sample - y))
    def pairwise_abs_diff(i, sample):
        return np.sum(np.abs(sample[i] - sample))
    term2 = 0.5 * np.sum(Parallel(n_jobs=-1)(delayed(pairwise_abs_diff)(i, sample) for i in range(M))) / M**2
    return term1 - term2

def crps_quantile_representation(y, sample):
    M = len(sample)
    sorted_sample = np.sort(sample)
    tau_values = np.linspace(1 / M, 1, M)
    crps_value = 0
    for k in range(M):
        x_k = sorted_sample[k]
        tau_k = tau_values[k]
        if y >= x_k:
            crps_value += tau_k * (y - x_k)
        else:
            crps_value += (1 - tau_k) * (x_k - y)
    return 2 * crps_value / M

def crps_closed_form_normal(y):
    phi_y = norm.pdf(y)
    Phi_y = norm.cdf(y)
    term1 = 2 * phi_y + y * (2 * Phi_y - 1)
    term2 = np.sqrt(1 / np.pi)
    return term1 - term2

# Generate a sample for approximations
sample_size = 5000
sample_normal = np.random.normal(size=sample_size)
sample_t = np.random.standard_t(df=3, size=sample_size)
sample_laplace = np.random.laplace(loc=0, scale=1, size=sample_size)

# Generate observations from a standard normal
y_values = np.random.normal(size=100)  # 100 observations from the standard normal
y_values = np.sort(y_values)  # Sort for smoother plotting

# Initialize lists for CRPS values
crps_abs_diff_normal = []
crps_quantile_normal = []
crps_closed_form_normal_values = []

crps_abs_diff_t = []
crps_quantile_t = []

crps_abs_diff_laplace = []
crps_quantile_laplace = []

# Compute CRPS for all methods
for y in y_values:
    # Normal distribution
    crps_abs_diff_normal.append(crps_absolute_differences_parallel(y, sample_normal))
    crps_quantile_normal.append(crps_quantile_representation(y, sample_normal))
    crps_closed_form_normal_values.append(crps_closed_form_normal(y))
    
    # t-distribution
    crps_abs_diff_t.append(crps_absolute_differences_parallel(y, sample_t))
    crps_quantile_t.append(crps_quantile_representation(y, sample_t))
    
    # Laplace distribution
    crps_abs_diff_laplace.append(crps_absolute_differences_parallel(y, sample_laplace))
    crps_quantile_laplace.append(crps_quantile_representation(y, sample_laplace))

# Plot the CRPS values for all methods
plt.figure(figsize=(12, 8))

# Standard Normal
plt.plot(y_values, crps_abs_diff_normal, label='Absolute Differences (Normal)', linestyle='--', color='blue')
plt.plot(y_values, crps_quantile_normal, label='Quantile Representation (Normal)', linestyle='-.', color='blue')
plt.plot(y_values, crps_closed_form_normal_values, label='Closed Form (Normal)', linestyle='-', color='blue')

# Student's t-distribution
plt.plot(y_values, crps_abs_diff_t, label='Absolute Differences (t-distribution)', linestyle='--', color='orange')
plt.plot(y_values, crps_quantile_t, label='Quantile Representation (t-distribution)', linestyle='-.', color='orange')

# Laplace distribution
plt.plot(y_values, crps_abs_diff_laplace, label='Absolute Differences (Laplace)', linestyle='--', color='green')
plt.plot(y_values, crps_quantile_laplace, label='Quantile Representation (Laplace)', linestyle='-.', color='green')

# Customize the plot
plt.title('Comparison of CRPS Methods Across Different Distributions', fontsize=14)
plt.xlabel('Observation Value (y)', fontsize=12)
plt.ylabel('CRPS Value', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True)

# Display the plot
plt.show()

# Compute the average CRPS for each method
average_crps_abs_diff_normal = np.mean(crps_abs_diff_normal)
average_crps_quantile_normal = np.mean(crps_quantile_normal)
average_crps_closed_form_normal = np.mean(crps_closed_form_normal_values)

average_crps_abs_diff_t = np.mean(crps_abs_diff_t)
average_crps_quantile_t = np.mean(crps_quantile_t)

average_crps_abs_diff_laplace = np.mean(crps_abs_diff_laplace)
average_crps_quantile_laplace = np.mean(crps_quantile_laplace)

# Create a DataFrame to display the results
average_results = pd.DataFrame({
    'Method': [
        'Absolute Differences (Normal)', 
        'Quantile Representation (Normal)', 
        'Closed Form (Normal)', 
        'Absolute Differences (t-distribution)', 
        'Quantile Representation (t-distribution)', 
        'Absolute Differences (Laplace)', 
        'Quantile Representation (Laplace)'
    ],
    'Average CRPS': [
        average_crps_abs_diff_normal, 
        average_crps_quantile_normal, 
        average_crps_closed_form_normal, 
        average_crps_abs_diff_t, 
        average_crps_quantile_t, 
        average_crps_abs_diff_laplace, 
        average_crps_quantile_laplace
    ]
})

# Print the table
print(average_results)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Function to calculate contributions to CRPS for a given observation and sample
def crps_contributions(y, sample):
    M = len(sample)
    sorted_sample = np.sort(sample)
    tau_values = np.linspace(1 / M, 1, M)  # Quantiles (x-axis)
    contributions = []  # Store contributions to CRPS (y-axis)
    
    for k in range(M):
        x_k = sorted_sample[k]
        tau_k = tau_values[k]
        if y >= x_k:
            crps_value = tau_k * (y - x_k)
        else:
            crps_value = (1 - tau_k) * (x_k - y)
        contributions.append(crps_value)
    
    return tau_values, contributions

# Generate a sample for demonstration
sample_size = 5000
sample_normal = np.random.normal(size=sample_size)

# Range of y_observations
y_observations = np.linspace(-3, 3, 20)  # 10 evenly spaced observations between -3 and 3

# Plot the contributions for multiple y_observations
plt.figure(figsize=(12, 8))

for y_observation in y_observations:
    tau_values, contributions = crps_contributions(y_observation, sample_normal)
    plt.plot(tau_values, contributions, label=f'y = {y_observation:.1f}', linewidth=1.5)

# Customize the plot
plt.title('CRPS Contributions by Quantile for Multiple Observations', fontsize=16)
plt.xlabel('Quantile (\u03C4)', fontsize=14)
plt.ylabel('Contribution to CRPS', fontsize=14)
plt.legend(fontsize=10, loc='upper right', title='Observations')
plt.grid(True)

# Display the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Function to calculate contributions to CRPS for a given observation and sample
def crps_contributions(y, sample):
    M = len(sample)
    sorted_sample = np.sort(sample)
    tau_values = np.linspace(1 / M, 1, M)  # Quantiles (x-axis)
    contributions = []  # Store contributions to CRPS (y-axis)
    
    for k in range(M):
        x_k = sorted_sample[k]
        tau_k = tau_values[k]
        if y >= x_k:
            crps_value = tau_k * (y - x_k)
        else:
            crps_value = (1 - tau_k) * (x_k - y)
        contributions.append(crps_value)
    
    return tau_values, contributions

# Generate a sample for demonstration
sample_size = 5000
sample_normal = np.random.normal(size=sample_size)

# Range of y_observations
y_observations = np.linspace(0, 3, 100)  # 10 evenly spaced observations between -3 and 3

# Plot the contributions for multiple y_observations with probability scaling
plt.figure(figsize=(12, 8))

for y_observation in y_observations:
    # Calculate contributions for the given observation
    tau_values, contributions = crps_contributions(y_observation, sample_normal)
    
    # Approximate the discrete probability for the observation using the normal CDF
    probability = norm.cdf(y_observation + 0.0001) - norm.cdf(y_observation - 0.0001)
    
    # Scale the contributions by the probability
    scaled_contributions = [c * probability for c in contributions]
    
    # Plot the scaled contributions
    plt.plot(tau_values, scaled_contributions, label=f'y = {y_observation:.1f}', linewidth=1.5)

# Customize the plot
plt.title('CRPS Contributions by Quantile for Multiple Observations (Scaled by Probability)', fontsize=16)
plt.xlabel('Quantile (\u03C4)', fontsize=14)
plt.ylabel('Contribution to CRPS (Scaled)', fontsize=14)
# plt.legend(fontsize=10, loc='upper right', title='Observations')
plt.grid(True)

# Display the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Function to calculate contributions to CRPS for a given observation and sample
def crps_contributions(y, sample):
    M = len(sample)
    sorted_sample = np.sort(sample)
    tau_values = np.linspace(1 / M, 1, M)  # Quantiles (x-axis)
    contributions = []  # Store contributions to CRPS (y-axis)
    
    for k in range(M):
        x_k = sorted_sample[k]
        tau_k = tau_values[k]
        if y >= x_k:
            crps_value = tau_k * (y - x_k)
        else:
            crps_value = (1 - tau_k) * (x_k - y)
        contributions.append(crps_value)
    
    return tau_values, contributions

# Generate a sample of y-values from a standard normal distribution
num_y_values = 100
y_observations = np.sort(np.random.normal(size=num_y_values))  # Sample and sort

# Generate a sample for the CRPS calculation
sample_size = 5000
sample_normal = np.random.normal(size=sample_size)

# Plot the contributions for multiple y_observations
plt.figure(figsize=(12, 8))

for i, y_observation in enumerate(y_observations):
    if y_observation > 0: 
        tau_values, contributions = crps_contributions(y_observation, sample_normal)
        plt.plot(tau_values, contributions, label=f'y = {y_observation:.2f}', linewidth=1.2, alpha=0.7)

# Customize the plot
plt.title('CRPS Contributions by Quantile for Sampled Observations', fontsize=16)
plt.xlabel('Quantile (\u03C4)', fontsize=14)
plt.ylabel('Contribution to CRPS', fontsize=14)
plt.grid(True)

# Display the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
T = 100  # Number of time steps
lambda_ = 0.9  # AR coefficient
b_t = 1.5  # Constant regression coefficient
V = 0.01  # Observation noise variance
W_z = 0.01  # State noise variance
p_jump = 0.1  # Probability of X_t being nonzero
jump_value = 10.0  # Positive jump value for X_t

# Initialize variables
z = np.zeros(T)
y = np.zeros(T)
X = np.random.choice([0, jump_value], size=T, p=[1 - p_jump, p_jump])

# Initial state
z[0] = np.random.normal(0, np.sqrt(W_z))

# Simulate the model
for t in range(1, T):
    eta_t = np.random.normal(0, np.sqrt(W_z))  # State noise
    eps_t = np.random.normal(0, np.sqrt(V))  # Observation noise
    
    # State equation
    z[t] = lambda_ * z[t - 1] + b_t * X[t] + eta_t
    
    # Observation equation
    y[t] = z[t] + eps_t

# Compute differences
delta_y = np.diff(y)
delta_z = np.diff(z)

# Plot the results
plt.figure(figsize=(12, 10))

# Observations
plt.subplot(4, 1, 1)
plt.plot(y, label="Observations (y_t)")
plt.title("Simulated Observations (y_t)")
plt.legend()

# Latent State
plt.subplot(4, 1, 2)
plt.plot(z, label="Latent State (z_t)", color='orange')
plt.title("Latent State (z_t)")
plt.legend()

# Covariate
plt.subplot(4, 1, 3)
plt.plot(X, label="Covariate (X_t)", color='green', linestyle='--')
plt.title("Covariate (X_t)")
plt.legend()

# Differences
plt.subplot(4, 1, 4)
plt.plot(delta_y, label="Difference in Observations (Δy_t)", linestyle='-')
plt.plot(delta_z, label="Difference in State (Δz_t)", linestyle='--', color='orange')
plt.title("Differences: Δy_t and Δz_t")
plt.legend()

plt.tight_layout()
plt.show()
